# مسودات تعليم الأسنان من صور الإنترنت

**قبل التشغيل:** من القائمة Runtime ← Change runtime type ← اختر **T4 GPU** ← Save.

**التشغيل:** Runtime ← **Run all**. سيطلب الإذن بالوصول إلى Google Drive، فوافق، لتُحفظ الصور والنتائج هناك.

- يستغرق نحو ساعة: تنزيل حتى 1500 صورة بترخيص حر من Openverse، ثم تعليمها تلقائياً.
- إذا انقطع الاتصال أو أُغلقت الصفحة، شغّل **Run all** مرة أخرى. يكمل من حيث توقف ولا يعيد تنزيل ما نزل.
- في النهاية يُنزَّل ملف `bootstrap-results.zip`. أرسله كما هو.


In [ ]:
N_PHOTOS = 1500      # عدد الصور المطلوب
USE_DRIVE = True     # حفظ العمل في Google Drive ليستأنف بعد أي انقطاع


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe', 'transformers>=4.56', 'onnxruntime'], check=True)
print('packages ready')


In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/smile_bootstrap'
else:
    WORK = os.path.abspath('smile_bootstrap')
os.makedirs(WORK, exist_ok=True)
print('working folder:', WORK)


In [ ]:
# the task code (bootstrap.py + our post-processing core), embedded so nothing else has to be uploaded
import base64, io, zipfile, shutil
PAYLOAD = '...'
CODE = os.path.join(WORK, 'code')
models = os.path.join(CODE, 'models')
keep = None
if os.path.isdir(models):                       # keep downloaded models between runs
    keep = os.path.join(WORK, '_models'); shutil.rmtree(keep, ignore_errors=True); shutil.move(models, keep)
shutil.rmtree(CODE, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall(CODE)
if keep: shutil.move(keep, models)
print(sorted(os.listdir(CODE)))


In [ ]:
# download (resumes) + automatic labelling; the log streams below
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU: slower but works)')
OUT = os.path.join(WORK, 'run2')
p = subprocess.Popen([sys.executable, '-u', 'bootstrap.py', '--openverse', str(N_PHOTOS), '--out', OUT], cwd=CODE,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Warning' not in line and 'warn(' not in line:
        print(line, end='')
p.wait()
print('exit code', p.returncode)


In [ ]:
# one zip to send back
import json
res = os.path.join(WORK, 'bootstrap-results.zip')
with zipfile.ZipFile(res, 'w', zipfile.ZIP_STORED) as z:
    for f in ('drafts.zip', 'report.json', 'log.txt'):
        if os.path.exists(os.path.join(OUT, f)):
            z.write(os.path.join(OUT, f), f)
    z.writestr('environment.json', json.dumps(dict(gpu=torch.cuda.is_available(), python=sys.version.split()[0], torch=torch.__version__)))
print(open(os.path.join(OUT, 'report.json')).read() if os.path.exists(os.path.join(OUT, 'report.json')) else 'no report')
if IN_COLAB:
    from google.colab import files
    files.download(res)
